   # Pergunta 1 — Qual UF mais cresceu em receita e por quê?
   Dados: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), pedidos de 2016 a 2018.

In [6]:
import duckdb
con = duckdb.connect("olist.duckdb")
con.sql("SHOW TABLES")

nameVARCHAR
clientes
itens
pagamentos
pedidos
produtos
reviews
vendedores


In [8]:
%load_ext sql
%sql con

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [10]:
%%sql
DESCRIBE pedidos

Running query in 'DuckDBPyConnection'

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
order_status,VARCHAR,YES,None,None,None
order_purchase_timestamp,TIMESTAMP,YES,None,None,None
order_approved_at,TIMESTAMP,YES,None,None,None
order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
order_delivered_customer_date,TIMESTAMP,YES,None,None,None
order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [14]:
%%sql SELECT MIN(order_purchase_timestamp), MAX(order_purchase_timestamp), COUNT(*)
FROM pedidos

Running query in 'DuckDBPyConnection'

min(order_purchase_timestamp),max(order_purchase_timestamp),count_star()
2016-09-04 21:15:19,2018-10-17 17:30:18,99441


In [15]:
%%sql
SELECT order_status, COUNT(*) AS qtd
FROM pedidos
GROUP BY order_status
ORDER BY qtd DESC

Running query in 'DuckDBPyConnection'

order_status,qtd
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [18]:
%%sql
DESCRIBE itens

Running query in 'DuckDBPyConnection'

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
order_item_id,BIGINT,YES,None,None,None
product_id,VARCHAR,YES,None,None,None
seller_id,VARCHAR,YES,None,None,None
shipping_limit_date,TIMESTAMP,YES,None,None,None
price,DOUBLE,YES,None,None,None
freight_value,DOUBLE,YES,None,None,None


In [19]:
%%sql
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS mes,
    COUNT(*) AS pedidos
FROM pedidos
WHERE order_status NOT IN ('canceled', 'unavailable')
GROUP BY mes
ORDER BY mes

Running query in 'DuckDBPyConnection'

mes,pedidos
2016-09-01 00:00:00,2
2016-10-01 00:00:00,293
2016-12-01 00:00:00,1
2017-01-01 00:00:00,787
2017-02-01 00:00:00,1718
2017-03-01 00:00:00,2617
2017-04-01 00:00:00,2377
2017-05-01 00:00:00,3640
2017-06-01 00:00:00,3205
2017-07-01 00:00:00,3946


In [20]:
%config SqlMagic.displaylimit = 50

In [21]:
%%sql
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS mes,
    COUNT(*) AS pedidos
FROM pedidos
WHERE order_status NOT IN ('canceled', 'unavailable')
GROUP BY mes
ORDER BY mes

Running query in 'DuckDBPyConnection'

mes,pedidos
2016-09-01 00:00:00,2
2016-10-01 00:00:00,293
2016-12-01 00:00:00,1
2017-01-01 00:00:00,787
2017-02-01 00:00:00,1718
2017-03-01 00:00:00,2617
2017-04-01 00:00:00,2377
2017-05-01 00:00:00,3640
2017-06-01 00:00:00,3205
2017-07-01 00:00:00,3946


In [22]:
%%sql
WITH base AS (
    SELECT
        p.order_id,
        c.customer_state AS uf,
        EXTRACT(YEAR FROM p.order_purchase_timestamp) AS ano,
        i.price
    FROM pedidos p
    JOIN clientes c ON c.customer_id = p.customer_id
    JOIN itens    i ON i.order_id    = p.order_id
    WHERE p.order_status NOT IN ('canceled', 'unavailable')
      AND EXTRACT(YEAR  FROM p.order_purchase_timestamp) IN (2017, 2018)
      AND EXTRACT(MONTH FROM p.order_purchase_timestamp) BETWEEN 1 AND 8
),
por_uf AS (
    SELECT uf, ano,
           SUM(price)               AS receita,
           COUNT(DISTINCT order_id) AS pedidos
    FROM base
    GROUP BY uf, ano
),
comparado AS (
    SELECT uf, ano, receita, pedidos,
           LAG(receita) OVER (PARTITION BY uf ORDER BY ano) AS receita_ant
    FROM por_uf
)
SELECT
    uf,
    ROUND(receita_ant, 2)                                  AS receita_2017,
    ROUND(receita, 2)                                      AS receita_2018,
    ROUND(receita - receita_ant, 2)                        AS cresc_abs,
    ROUND(100.0 * (receita - receita_ant) / receita_ant, 1) AS cresc_pct,
    pedidos                                                AS pedidos_2018
FROM comparado
WHERE ano = 2018 AND receita_ant IS NOT NULL
ORDER BY cresc_abs DESC

Running query in 'DuckDBPyConnection'

uf,receita_2017,receita_2018,cresc_abs,cresc_pct,pedidos_2018
SP,1111718.45,2960326.25,1848607.8,166.3,23598
MG,337167.1,851003.48,513836.38,152.4,6134
RJ,442351.03,901277.72,458926.69,103.7,6516
PR,161448.68,380766.31,219317.63,135.8,2733
RS,181061.83,382652.74,201590.91,111.3,2764
SC,121177.14,284036.71,162859.57,134.4,1896
BA,127864.79,272158.01,144293.22,112.8,1770
DF,63341.95,166262.04,102920.09,162.5,1210
ES,60263.83,153824.97,93561.14,155.3,1055
PE,54303.71,137775.88,83472.17,153.7,872


In [23]:
%%sql
WITH base AS (
    SELECT p.order_id, c.customer_state AS uf,
           EXTRACT(YEAR FROM p.order_purchase_timestamp) AS ano, i.price
    FROM pedidos p
    JOIN clientes c ON c.customer_id = p.customer_id
    JOIN itens    i ON i.order_id    = p.order_id
    WHERE p.order_status NOT IN ('canceled', 'unavailable')
      AND EXTRACT(YEAR  FROM p.order_purchase_timestamp) IN (2017, 2018)
      AND EXTRACT(MONTH FROM p.order_purchase_timestamp) BETWEEN 1 AND 8
),
por_uf AS (
    SELECT uf, ano,
           SUM(price)                            AS receita,
           COUNT(DISTINCT order_id)              AS pedidos,
           SUM(price) / COUNT(DISTINCT order_id) AS ticket
    FROM base
    GROUP BY uf, ano
)
SELECT
    a.uf,
    a.pedidos AS pedidos_2017, b.pedidos AS pedidos_2018,
    ROUND(100.0 * (b.pedidos - a.pedidos) / a.pedidos, 1) AS cresc_pedidos_pct,
    ROUND(a.ticket, 2) AS ticket_2017, ROUND(b.ticket, 2) AS ticket_2018,
    ROUND(100.0 * (b.ticket - a.ticket) / a.ticket, 1)    AS cresc_ticket_pct
FROM por_uf a
JOIN por_uf b ON a.uf = b.uf AND a.ano = 2017 AND b.ano = 2018
WHERE b.pedidos >= 200
ORDER BY b.receita DESC

Running query in 'DuckDBPyConnection'

uf,pedidos_2017,pedidos_2018,cresc_pedidos_pct,ticket_2017,ticket_2018,cresc_ticket_pct
SP,8782,23598,168.7,126.59,125.45,-0.9
RJ,3063,6516,112.7,144.42,138.32,-4.2
MG,2664,6134,130.3,126.56,138.74,9.6
RS,1410,2764,96.0,128.41,138.44,7.8
PR,1213,2733,125.3,133.1,139.32,4.7
SC,876,1896,116.4,138.33,149.81,8.3
BA,799,1770,121.5,160.03,153.76,-3.9
DF,420,1210,188.1,150.81,137.41,-8.9
ES,485,1055,117.5,124.26,145.81,17.3
GO,466,1048,124.9,148.31,143.46,-3.3


## Conclusão

- **São Paulo** foi o estado que mais cresceu em receita: +1,85 milhão de reais entre jan–ago/2017 e jan–ago/2018 (+166%), respondendo por 40% de todo o crescimento da plataforma. Entre os cinco maiores estados, **Minas Gerais** teve o maior crescimento percentual (+152%, de 337 mil para 851 mil reais).
- O crescimento veio de **volume, não de preço**: SP fez +169% em pedidos com ticket parado (−0,9%). MG foi a exceção entre os grandes, crescendo nas duas pontas (+130% em pedidos, +9,6% em ticket, de 126 para 139 reais). Nos estados de menor volume o ticket é mais alto: PA 186, MT 162, PB 231 reais.
- A plataforma inteira cresceu entre 100% e 160% no período, porque 2017 ainda era rampa de implantação. O que destaca uma UF é crescer acima dessa faixa. 2016 foi excluído (menos de 300 pedidos no total) e set–out/2018 também (dados incompletos); por isso o recorte jan–ago.
- **Decisões:** excluí pedidos `canceled` (625) e `unavailable` (609), ~1,2% do total, porque não geram receita. Na comparação de ticket, considerei só estados com 200+ pedidos em 2018, porque percentual em base pequena é ruído (RR "cresceu 609%" partindo de 877 reais).
- **Hipótese não confirmada:** PA, MT e PB têm ticket de 160 a 230 reais com poucos pedidos. Pode ser que onde o frete pesa o cliente compre menos vezes e produtos mais caros. Para confirmar seria preciso cruzar com `freight_value` e categoria de produto.